# Training di RTX 5070 Ti

Notebook ini berisi semua training final penelitian. Semua run dijalankan di satu GPU, yaitu NVIDIA GeForce RTX 5070 Ti, supaya waktu training, latensi, dan memori bisa dibandingkan dengan adil. Training lama di RTX 5060 Ti (notebook 05) tetap disimpan sebagai arsip, tetapi tidak dipakai untuk hasil akhir.

Ada tiga skenario yang dijalankan:

| Skenario | Yang diuji | Kode di nama run |
|---|---|---|
| S1 | 2 model × 3 input × 4 learning rate (1e-5, 2e-5, 3e-5, 5e-5), seed 42 | R1, R2, R3 |
| S2 | jumlah layer teratas yang ikut dilatih, dengan input early fusion | P2 |
| S3 | konfigurasi terbaik S1 diulang dengan seed 43 dan 44 | S43, S44 |

Seed adalah angka awal untuk pengacak. Seed yang berbeda membuat urutan data saat training dan bobot awal lapisan klasifikasi ikut berbeda. Kalau hasil dengan beberapa seed hampir sama, berarti hasilnya bukan kebetulan.

Pengaturan yang sama untuk semua run:

| Pengaturan | Nilai |
|---|---|
| Panjang maksimal input | 256 token |
| Batch size | 32 |
| Epoch | 3 |
| Optimizer | AdamW |
| Warmup | 10% langkah pertama, lalu learning rate turun linear sampai 0 |
| Weight decay | 0,01 |
| Presisi | BF16 |
| Pemilihan epoch terbaik | macro-F1 validasi |

Data test tidak dibuka di notebook ini. Prediksi test setiap run disimpan di folder `test_sealed/` dan baru dihitung di notebook 07 setelah konfigurasi final dipilih dari hasil validasi.

## 1. Setup

Bagian ini menyiapkan path folder dan memastikan GPU yang dipakai memang RTX 5070 Ti. Kalau GPU-nya berbeda, sel ini sengaja berhenti dengan error supaya hasil dari dua GPU tidak tercampur di satu folder.

In [5]:
import sys, importlib
from pathlib import Path

import pandas as pd
import torch
import transformers

ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT / "src"))
import training as T
importlib.reload(T)
transformers.logging.set_verbosity_error()

DATA_DIR = ROOT / "data/tokenized_full"
OUT_DIR = ROOT / "data/training_rtx5070ti"
PILOT_DIR = OUT_DIR / "pilot"
FULL_DIR = OUT_DIR / "full"
PILOT = dict(train_split="train_pilot", val_limit=1000, test_limit=1000, epochs=2)

assert "5070 Ti" in torch.cuda.get_device_name(), "GPU bukan RTX 5070 Ti, hasil jangan dicampur"
free, total = torch.cuda.mem_get_info()
print("GPU:", torch.cuda.get_device_name(), f"| VRAM terpakai {(total - free) / 1e9:.1f} / {total / 1e9:.1f} GB")
print("torch", torch.__version__, "| transformers", transformers.__version__, "| BF16:", torch.cuda.is_bf16_supported())

GPU: NVIDIA GeForce RTX 5070 Ti | VRAM terpakai 1.7 / 17.1 GB
torch 2.11.0+cu128 | transformers 5.17.0 | BF16: True


#### Insight
GPU yang terbaca adalah NVIDIA GeForce RTX 5070 Ti dengan memori 17,1 GB, dan sebelum training baru terpakai 1,8 GB untuk tampilan layar dan aplikasi lain. PyTorch yang dipakai versi 2.11.0 dengan CUDA 12.8 (cu128), versi yang memang dibutuhkan kartu seri RTX 50. BF16 didukung, jadi pengaturan presisi sama dengan training sebelumnya.

## 2. Pilot di PC baru

Bagian ini menjalankan pilot kecil (2.000 data latih, 1.000 data validasi, 2 epoch) untuk memastikan kode training berjalan normal di PC baru sebelum training data penuh.

In [12]:
for cfg in T.build_queue([0]):
    T.run_experiment(cfg, DATA_DIR, PILOT_DIR, **PILOT)

[R0_comment_roberta_lr3e-05_s42] sudah selesai, dilewati
[R0_comment_distilbert_lr3e-05_s42] sudah selesai, dilewati
[R0_early_roberta_lr3e-05_s42] sudah selesai, dilewati
[R0_early_distilbert_lr3e-05_s42] sudah selesai, dilewati
[R0_late_roberta_lr3e-05_s42] sudah selesai, dilewati
[R0_late_distilbert_lr3e-05_s42] sudah selesai, dilewati


#### Insight
Keenam run pilot tertulis "sudah selesai, dilewati" karena pilot sudah pernah dijalankan sebelumnya. Kode training selalu mengecek `runs.csv` dulu, dan run yang namanya sudah ada tidak dilatih ulang. Jadi sel ini aman dijalankan berkali-kali.

## 3. Membandingkan pilot di dua GPU

Bagian ini membandingkan hasil pilot di RTX 5060 Ti dan RTX 5070 Ti. Tujuannya untuk melihat apakah pindah GPU mengubah hasil, dan seberapa cepat GPU yang baru. Tabel pertama berisi metrik validasi, tabel kedua berisi perbandingan kecepatan dan memori (angka 5070 Ti dibagi angka 5060 Ti).

In [13]:
cols = ["val_accuracy", "val_precision", "val_recall", "val_f1", "val_macro_f1"]
old = pd.read_csv(ROOT / "data/training_rtx5060ti/pilot/runs.csv").set_index("run_id")
new = pd.read_csv(PILOT_DIR / "runs.csv").set_index("run_id")
cmp = pd.concat({"5060 Ti": old[cols], "5070 Ti": new[cols]}, axis=1)
cmp[("selisih", "macro_f1")] = cmp[("5070 Ti", "val_macro_f1")] - cmp[("5060 Ti", "val_macro_f1")]
display(cmp.round(4))

eff = ["train_samples_per_sec", "latency_bs1_ms", "latency_bs32_ms", "peak_mem_gb"]
display((new[eff] / old[eff]).round(2).add_suffix(" (5070/5060)"))


5060 Ti                           \
                                  val_accuracy val_precision val_recall   
run_id                                                                    
R0_comment_roberta_lr3e-05_s42           0.655        0.6807      0.584   
R0_comment_distilbert_lr3e-05_s42        0.640        0.6724      0.546   
R0_early_roberta_lr3e-05_s42             0.635        0.6497      0.586   
R0_early_distilbert_lr3e-05_s42          0.605        0.6031      0.614   
R0_late_roberta_lr3e-05_s42              0.576        0.5739      0.590   
R0_late_distilbert_lr3e-05_s42           0.640        0.6306      0.676   

                                                            5070 Ti  \
                                   val_f1 val_macro_f1 val_accuracy   
run_id                                                                
R0_comment_roberta_lr3e-05_s42     0.6286       0.6533        0.664   
R0_comment_distilbert_lr3e-05_s42  0.6026       0.6368        0.629   
R0_early_roberta_lr3e-05_s42       0.6162       0.6341        0.649   
R0_early_distilbert_lr3e-05_s42    0.6085       0.6050        0.606   
R0_late_roberta_lr3e-05_s42        0.5819       0.5759        0.655   
R0_late_distilbert_lr3e-05_s42     0.6525       0.6395        0.636   

                                                                    \
                                  val_precision val_recall  val_f1   
run_id                                                               
R0_comment_roberta_lr3e-05_s42           0.6608      0.674  0.6673   
R0_comment_distilbert_lr3e-05_s42        0.6554      0.544  0.5945   
R0_early_roberta_lr3e-05_s42             0.6987      0.524  0.5989   
R0_early_distilbert_lr3e-05_s42          0.5953      0.662  0.6269   
R0_late_roberta_lr3e-05_s42              0.7183      0.510  0.5965   
R0_late_distilbert_lr3e-05_s42           0.6283      0.666  0.6466   

                                                selisih  
                                  val_macro_f1 macro_f1  
run_id                                                   
R0_comment_roberta_lr3e-05_s42          0.6640   0.0107  
R0_comment_distilbert_lr3e-05_s42       0.6263  -0.0105  
R0_early_roberta_lr3e-05_s42            0.6434   0.0093  
R0_early_distilbert_lr3e-05_s42         0.6048  -0.0002  
R0_late_roberta_lr3e-05_s42             0.6476   0.0717  
R0_late_distilbert_lr3e-05_s42          0.6357  -0.0039

,train_samples_per_sec (5070/5060),latency_bs1_ms (5070/5060),latency_bs32_ms (5070/5060),peak_mem_gb (5070/5060)
run_id,,,,
R0_comment_roberta_lr3e-05_s42,1.86,0.57,0.55,1.00
R0_comment_distilbert_lr3e-05_s42,1.90,0.54,0.51,1.89
R0_early_roberta_lr3e-05_s42,1.84,0.53,0.50,1.30
R0_early_distilbert_lr3e-05_s42,1.81,0.53,0.50,1.57
R0_late_roberta_lr3e-05_s42,1.81,0.55,0.52,1.23
R0_late_distilbert_lr3e-05_s42,1.79,0.54,0.52,1.45


#### Insight
Hasil pilot di dua GPU tidak persis sama. Selisih macro-F1 berkisar dari -0,0105 sampai +0,0717. Ini wajar karena data pilot sangat kecil. Dengan hanya 1.000 data validasi, satu atau dua tebakan yang berbeda sudah menggeser nilai cukup jauh, dan model yang dilatih dengan 2.000 data juga belum stabil. Contohnya late fusion RoBERTa di 5060 Ti hanya mendapat 0,5759, hampir sama dengan menebak acak, jadi run seperti ini mudah berubah. Hasil juga tidak bisa identik sampai digit terakhir karena jumlah inti kedua GPU berbeda, sehingga urutan penjumlahan angka di dalam GPU sedikit berbeda.

Dari sisi kecepatan, RTX 5070 Ti memproses data latih 1,79 sampai 1,90 kali lebih cepat, dan waktu menebaknya hanya 0,50 sampai 0,57 kali waktu di 5060 Ti, artinya sekitar 2 kali lebih cepat.

Rasio memori (1,00 sampai 1,89) tidak masuk akal, karena memori yang dipakai model seharusnya tidak bergantung pada kecepatan GPU. Run pertama sama persis (1,00), tetapi run berikutnya selalu lebih besar sekitar 1,5 GB. Penyebabnya sesi notebook waktu itu masih menyimpan sisa model dari run sebelumnya. Karena itu angka memori pilot di 5070 Ti tidak dipakai. Masalah ini diatasi dengan restart kernel dan pengecekan sisa memori setelah setiap run di bagian 6. Di training penuh, memori RoBERTa comment saja tercatat 5,09 GB, sama dengan di 5060 Ti.

## 4. Perkiraan waktu training penuh

Bagian ini memperkirakan lama training data penuh dari kecepatan pilot di epoch kedua, supaya jadwal training bisa direncanakan. Waktu dihitung untuk training 3 epoch, validasi setiap epoch, dan prediksi data test.

In [14]:
N_TRAIN, N_VAL, N_TEST, EPOCHS = 799_459, 100_276, 101_129, 3
hist = pd.read_csv(PILOT_DIR / "history.csv")
last = hist.sort_values("epoch").groupby("run_id").tail(1)          # epoch ke-2 (GPU sudah "panas")
est = last.merge(new.reset_index()[["run_id", "input", "model", "test_pred_min", "n_test"]], on="run_id")
est["train_mnt_per_epoch"] = N_TRAIN / est["train_samples_per_sec"] / 60
est["val_mnt_per_epoch"] = N_VAL / est["val_samples_per_sec"] / 60
est["test_mnt"] = est["test_pred_min"] * N_TEST / est["n_test"]
est["total_jam"] = (EPOCHS * (est["train_mnt_per_epoch"] + est["val_mnt_per_epoch"]) + est["test_mnt"]) / 60
display(est[["input", "model", "train_mnt_per_epoch", "val_mnt_per_epoch", "test_mnt", "total_jam"]].round(2))
print(f"Perkiraan 1 seed (6 konfigurasi x {EPOCHS} epoch): {est['total_jam'].sum():.1f} jam")


,input,model,train_mnt_per_epoch,val_mnt_per_epoch,test_mnt,total_jam
0,comment,distilbert,8.87,0.12,0.11,0.45
1,comment,roberta,17.32,0.21,0.20,0.88
2,early,distilbert,10.45,0.21,0.22,0.54
3,early,roberta,20.12,0.39,0.43,1.03
4,late,roberta,29.51,0.59,0.63,1.52
5,late,distilbert,15.48,0.34,0.33,0.80


Perkiraan 1 seed (6 konfigurasi x 3 epoch): 5.2 jam


#### Insight
Satu putaran 6 konfigurasi (3 input × 2 model) diperkirakan butuh 5,2 jam. Yang paling lama late fusion RoBERTa (1,52 jam) dan yang paling cepat comment saja DistilBERT (0,45 jam). Karena S1 punya 4 learning rate, 24 run S1 diperkirakan butuh sekitar 4 × 5,2 = 20,8 jam.

Perkiraan ini cukup tepat. Di bagian 7, waktu training 3 epoch RoBERTa early fusion tercatat sekitar 55 sampai 58 menit, dekat dengan perkiraan 1,03 jam yang juga sudah termasuk validasi dan prediksi test.

## 5. Daftar run

Bagian ini menyusun antrean semua run dan menandai mana yang sudah selesai.

`BEST_LR` berisi learning rate terbaik setiap kombinasi input dan model, diambil dari hasil S1 di GPU ini (lihat bagian 7). Fungsi `queue_p2` membuat run S2, yaitu early fusion dengan 0, 3, atau 6 layer teratas yang dilatih. Untuk 0 layer dipakai learning rate 1e-4 dan 1e-3, karena yang dilatih hanya lapisan klasifikasi yang kecil. Untuk 3 dan 6 layer dipakai learning rate terbaik early fusion. Fungsi `queue_seed` mengulang 6 konfigurasi terbaik dengan seed lain.

Nama run dibaca seperti ini, contohnya `P2_early_roberta_L3_lr2e-05_s42`: skenario S2 (P2), input early fusion, model RoBERTa, 3 layer teratas dilatih (L3), learning rate 2e-5, seed 42.

In [2]:
BEST_LR = {("comment", "roberta"): 1e-5, ("comment", "distilbert"): 3e-5,
           ("early", "roberta"): 2e-5, ("early", "distilbert"): 1e-5,
           ("late", "roberta"): 1e-5, ("late", "distilbert"): 1e-5}



def queue_p2(seed=42):
    """Jumlah layer teratas yang dilatih (early fusion). Layer 0 = hanya head, perlu LR lebih besar."""
    q = []
    for model, layers in [("roberta", [0, 3, 6]), ("distilbert", [0, 3])]:
        for k in layers:
            for lr in ([1e-4, 1e-3] if k == 0 else [BEST_LR[("early", model)]]):
                q.append({"run_id": f"P2_early_{model}_L{k}_lr{lr:.0e}_s{seed}", "round": "P2",
                          "input": "early", "model": model, "lr": lr, "seed": seed, "n_train_layers": k})
    return q


def queue_seed(seed):
    """6 konfigurasi terbaik diulang dengan seed lain."""
    return [{"run_id": f"S{seed}_{inp}_{model}_lr{lr:.0e}_s{seed}", "round": f"S{seed}",
             "input": inp, "model": model, "lr": lr, "seed": seed}
            for (inp, model), lr in BEST_LR.items()]



QUEUE = (T.build_queue([1, 2, 3]) + queue_p2() + queue_seed(43) + queue_seed(44)
         + [c for s in (43, 44) for c in queue_p2(s) if c["n_train_layers"] > 0])
T.run_status(QUEUE, FULL_DIR)


,run_id,round,input,model,lr,seed,n_train_layers,status
0,R1_comment_roberta_lr3e-05_s42,1,comment,roberta,0.00003,42,NaN,selesai
1,R1_comment_distilbert_lr3e-05_s42,1,comment,distilbert,0.00003,42,NaN,selesai
2,R1_early_roberta_lr3e-05_s42,1,early,roberta,0.00003,42,NaN,selesai
3,R1_early_distilbert_lr3e-05_s42,1,early,distilbert,0.00003,42,NaN,selesai
4,R1_late_roberta_lr3e-05_s42,1,late,roberta,0.00003,42,NaN,selesai
5,R1_late_distilbert_lr3e-05_s42,1,late,distilbert,0.00003,42,NaN,selesai
6,R2_early_roberta_lr1e-05_s42,2,early,roberta,0.00001,42,NaN,selesai
7,R2_early_distilbert_lr1e-05_s42,2,early,distilbert,0.00001,42,NaN,selesai
8,R2_early_roberta_lr2e-05_s42,2,early,roberta,0.00002,42,NaN,selesai
9,R2_early_distilbert_lr2e-05_s42,2,early,distilbert,0.00002,42,NaN,selesai


#### Insight
Antrean berisi 49 run. Saat sel ini dijalankan, 32 run sudah selesai dan 17 run belum. Ke-17 run itu muncul karena learning rate terbaik dari S1 di GPU ini berbeda dari hasil di RTX 5060 Ti untuk 4 konfigurasi (comment RoBERTa, comment DistilBERT, early RoBERTa, early DistilBERT). Supaya seed 42, 43, dan 44 memakai learning rate yang sama, seed 43 dan 44 untuk keempat konfigurasi itu dijalankan ulang. S2 juga dijalankan ulang dengan learning rate terbaik early fusion yang baru, sekaligus ditambah seed 43 dan 44.

Run lama yang memakai learning rate lama (11 run) tidak dihapus dari `runs.csv`, hanya tidak dipakai untuk hasil akhir. Namanya berbeda karena learning rate ikut tertulis di nama run, jadi tidak ada yang tertimpa.

## 6. Training

Bagian ini menjalankan semua run di antrean. Run yang sudah selesai otomatis dilewati. Setelah setiap run, sisa memori GPU dicek. Kalau masih ada sisa lebih dari 0,5 GB, training berhenti supaya angka memori run berikutnya tidak ikut tercemar.

In [16]:
for cfg in QUEUE:
    T.run_experiment(cfg, DATA_DIR, FULL_DIR)
    sisa = torch.cuda.memory_allocated() / 1e9
    print(f"sisa memori GPU setelah run: {sisa:.2f} GB")
    assert sisa < 0.5, "Memori run sebelumnya tidak terlepas: restart kernel, lalu jalankan sel 1, 5, 6"


[R1_comment_roberta_lr3e-05_s42] sudah selesai, dilewati
sisa memori GPU setelah run: 0.02 GB
[R1_comment_distilbert_lr3e-05_s42] sudah selesai, dilewati
sisa memori GPU setelah run: 0.02 GB
[R1_early_roberta_lr3e-05_s42] sudah selesai, dilewati
sisa memori GPU setelah run: 0.02 GB
[R1_early_distilbert_lr3e-05_s42] sudah selesai, dilewati
sisa memori GPU setelah run: 0.02 GB
[R1_late_roberta_lr3e-05_s42] sudah selesai, dilewati
sisa memori GPU setelah run: 0.02 GB
[R1_late_distilbert_lr3e-05_s42] sudah selesai, dilewati
sisa memori GPU setelah run: 0.02 GB
[R2_early_roberta_lr1e-05_s42] sudah selesai, dilewati
sisa memori GPU setelah run: 0.02 GB
[R2_early_distilbert_lr1e-05_s42] sudah selesai, dilewati
sisa memori GPU setelah run: 0.02 GB
[R2_early_roberta_lr2e-05_s42] sudah selesai, dilewati
sisa memori GPU setelah run: 0.02 GB
[R2_early_distilbert_lr2e-05_s42] sudah selesai, dilewati
sisa memori GPU setelah run: 0.02 GB
[R2_early_roberta_lr5e-05_s42] sudah selesai, dilewati
sisa mem

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1185.34it/s]


[P2_early_roberta_L3_lr2e-05_s42] train 799,459 | val 100,276 | 24,984 langkah/epoch | parameter 124.6 juta (dilatih 21.9 juta)
  epoch 1 |  2,499/24,984 | loss 0.6434 | lr 6.67e-06 | 0.8 mnt, sisa ±7.1 mnt
  epoch 1 |  4,997/24,984 | loss 0.5585 | lr 1.33e-05 | 1.6 mnt, sisa ±6.2 mnt
  epoch 1 |  7,496/24,984 | loss 0.5374 | lr 2.00e-05 | 2.4 mnt, sisa ±5.5 mnt
  epoch 1 |  9,994/24,984 | loss 0.5259 | lr 1.93e-05 | 3.1 mnt, sisa ±4.7 mnt
  epoch 1 | 12,492/24,984 | loss 0.5123 | lr 1.85e-05 | 3.9 mnt, sisa ±3.9 mnt
  epoch 1 | 14,991/24,984 | loss 0.5007 | lr 1.78e-05 | 4.7 mnt, sisa ±3.1 mnt
  epoch 1 | 17,489/24,984 | loss 0.5011 | lr 1.70e-05 | 5.5 mnt, sisa ±2.4 mnt
  epoch 1 | 19,988/24,984 | loss 0.4932 | lr 1.63e-05 | 6.3 mnt, sisa ±1.6 mnt
  epoch 1 | 22,486/24,984 | loss 0.4888 | lr 1.56e-05 | 7.0 mnt, sisa ±0.8 mnt
  epoch 1 | 24,984/24,984 | loss 0.4872 | lr 1.48e-05 | 7.7 mnt, sisa ±0.0 mnt
  epoch 1 selesai | train 7.7 mnt | val loss 0.4748 | acc 0.7730 P 0.7792 R 0.7540

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1843.71it/s]


[P2_early_roberta_L6_lr2e-05_s42] train 799,459 | val 100,276 | 24,984 langkah/epoch | parameter 124.6 juta (dilatih 43.1 juta)
  epoch 1 |  2,499/24,984 | loss 0.6205 | lr 6.67e-06 | 1.1 mnt, sisa ±9.5 mnt
  epoch 1 |  4,997/24,984 | loss 0.5411 | lr 1.33e-05 | 2.1 mnt, sisa ±8.4 mnt
  epoch 1 |  7,496/24,984 | loss 0.5210 | lr 2.00e-05 | 3.1 mnt, sisa ±7.3 mnt
  epoch 1 |  9,994/24,984 | loss 0.5089 | lr 1.93e-05 | 4.2 mnt, sisa ±6.3 mnt
  epoch 1 | 12,492/24,984 | loss 0.4956 | lr 1.85e-05 | 5.2 mnt, sisa ±5.2 mnt
  epoch 1 | 14,991/24,984 | loss 0.4854 | lr 1.78e-05 | 6.3 mnt, sisa ±4.2 mnt
  epoch 1 | 17,489/24,984 | loss 0.4842 | lr 1.70e-05 | 7.3 mnt, sisa ±3.1 mnt
  epoch 1 | 19,988/24,984 | loss 0.4766 | lr 1.63e-05 | 8.4 mnt, sisa ±2.1 mnt
  epoch 1 | 22,486/24,984 | loss 0.4720 | lr 1.56e-05 | 9.4 mnt, sisa ±1.0 mnt
  epoch 1 | 24,984/24,984 | loss 0.4725 | lr 1.48e-05 | 10.5 mnt, sisa ±0.0 mnt
  epoch 1 selesai | train 10.5 mnt | val loss 0.4643 | acc 0.7821 P 0.7854 R 0.76

...

[P2_early_roberta_L3_lr2e-05_s44] train 799,459 | val 100,276 | 24,984 langkah/epoch | parameter 124.6 juta (dilatih 21.9 juta)
  epoch 1 |  2,499/24,984 | loss 0.6447 | lr 6.67e-06 | 0.7 mnt, sisa ±6.7 mnt
  epoch 1 |  4,997/24,984 | loss 0.5575 | lr 1.33e-05 | 1.5 mnt, sisa ±6.0 mnt
  epoch 1 |  7,496/24,984 | loss 0.5387 | lr 2.00e-05 | 2.2 mnt, sisa ±5.2 mnt
  epoch 1 |  9,994/24,984 | loss 0.5224 | lr 1.93e-05 | 3.0 mnt, sisa ±4.5 mnt
  epoch 1 | 12,492/24,984 | loss 0.5110 | lr 1.85e-05 | 3.7 mnt, sisa ±3.7 mnt
  epoch 1 | 14,991/24,984 | loss 0.5057 | lr 1.78e-05 | 4.5 mnt, sisa ±3.0 mnt
  epoch 1 | 17,489/24,984 | loss 0.4998 | lr 1.70e-05 | 5.2 mnt, sisa ±2.2 mnt
  epoch 1 | 19,988/24,984 | loss 0.4953 | lr 1.63e-05 | 6.0 mnt, sisa ±1.5 mnt
  epoch 1 | 22,486/24,984 | loss 0.4920 | lr 1.56e-05 | 6.7 mnt, sisa ±0.7 mnt
  epoch 1 | 24,984/24,984 | loss 0.4870 | lr 1.48e-05 | 7.5 mnt, sisa ±0.0 mnt
  epoch 1 selesai | train 7.5 mnt | val loss 0.4747 | acc 0.7731 P 0.7729 R 0.7654

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1788.89it/s]


[P2_early_roberta_L6_lr2e-05_s44] train 799,459 | val 100,276 | 24,984 langkah/epoch | parameter 124.6 juta (dilatih 43.1 juta)
  epoch 1 |  2,499/24,984 | loss 0.6234 | lr 6.67e-06 | 1.1 mnt, sisa ±9.5 mnt
  epoch 1 |  4,997/24,984 | loss 0.5389 | lr 1.33e-05 | 2.1 mnt, sisa ±8.4 mnt
  epoch 1 |  7,496/24,984 | loss 0.5223 | lr 2.00e-05 | 3.2 mnt, sisa ±7.4 mnt
  epoch 1 |  9,994/24,984 | loss 0.5066 | lr 1.93e-05 | 4.2 mnt, sisa ±6.3 mnt
  epoch 1 | 12,492/24,984 | loss 0.4947 | lr 1.85e-05 | 5.3 mnt, sisa ±5.3 mnt
  epoch 1 | 14,991/24,984 | loss 0.4894 | lr 1.78e-05 | 6.3 mnt, sisa ±4.2 mnt
  epoch 1 | 17,489/24,984 | loss 0.4834 | lr 1.70e-05 | 7.4 mnt, sisa ±3.2 mnt
  epoch 1 | 19,988/24,984 | loss 0.4782 | lr 1.63e-05 | 8.4 mnt, sisa ±2.1 mnt
  epoch 1 | 22,486/24,984 | loss 0.4760 | lr 1.56e-05 | 9.4 mnt, sisa ±1.0 mnt
  epoch 1 | 24,984/24,984 | loss 0.4716 | lr 1.48e-05 | 10.5 mnt, sisa ±0.0 mnt
  epoch 1 selesai | train 10.5 mnt | val loss 0.4601 | acc 0.7831 P 0.7989 R 0.74

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2156.47it/s]


[P2_early_distilbert_L3_lr1e-05_s44] train 799,459 | val 100,276 | 24,984 langkah/epoch | parameter 65.8 juta (dilatih 21.9 juta)
  epoch 1 |  2,499/24,984 | loss 0.6633 | lr 3.33e-06 | 0.6 mnt, sisa ±5.0 mnt
  epoch 1 |  4,997/24,984 | loss 0.5939 | lr 6.67e-06 | 1.1 mnt, sisa ±4.4 mnt
  epoch 1 |  7,496/24,984 | loss 0.5698 | lr 1.00e-05 | 1.6 mnt, sisa ±3.8 mnt
  epoch 1 |  9,994/24,984 | loss 0.5548 | lr 9.63e-06 | 2.2 mnt, sisa ±3.3 mnt
  epoch 1 | 12,492/24,984 | loss 0.5456 | lr 9.26e-06 | 2.7 mnt, sisa ±2.7 mnt
  epoch 1 | 14,991/24,984 | loss 0.5400 | lr 8.89e-06 | 3.3 mnt, sisa ±2.2 mnt
  epoch 1 | 17,489/24,984 | loss 0.5318 | lr 8.52e-06 | 3.8 mnt, sisa ±1.6 mnt
  epoch 1 | 19,988/24,984 | loss 0.5307 | lr 8.15e-06 | 4.4 mnt, sisa ±1.1 mnt
  epoch 1 | 22,486/24,984 | loss 0.5206 | lr 7.78e-06 | 4.9 mnt, sisa ±0.5 mnt
  epoch 1 | 24,984/24,984 | loss 0.5168 | lr 7.41e-06 | 5.5 mnt, sisa ±0.0 mnt
  epoch 1 selesai | train 5.5 mnt | val loss 0.5163 | acc 0.7423 P 0.7323 R 0.75

#### Insight
Ke-17 run baru selesai tanpa error, dan sisa memori setelah setiap run hanya 0,02 GB. Artinya memori dari run sebelumnya selalu terlepas dan angka memori puncak setiap run bisa dipercaya.

Beberapa hal yang terlihat dari log:
1. Membekukan layer membuat training jauh lebih ringan. RoBERTa dengan 3 layer teratas hanya melatih 21,9 juta dari 124,6 juta parameter, butuh 7,4 menit per epoch dan memori 1,67 GB. RoBERTa early fusion yang dilatih penuh butuh 18,9 menit per epoch dan 5,11 GB.
2. DistilBERT comment saja dengan learning rate 3e-5 mulai overfitting di epoch 3. Loss validasi naik dari 0,4920 ke 0,5243 (seed 43) dan dari 0,4875 ke 0,5250 (seed 44), jadi epoch terbaiknya epoch 2. Overfitting artinya model mulai menghafal data latih sehingga makin ragu pada data baru.
3. RoBERTa early fusion seed 44 juga terbaik di epoch 2 (macro-F1 0,7965), sedikit di atas epoch 3 (0,7962), dan loss validasinya naik dari 0,4430 ke 0,4702.
4. Gaya menebak model bisa bergeser antar-epoch walaupun nilainya mirip. Pada RoBERTa early fusion seed 43, epoch 1 punya precision 0,8131 dan recall 0,7335 (jarang menebak sarkas), sedangkan epoch 3 punya precision 0,7869 dan recall 0,8072 (lebih sering menebak sarkas).

## 7. Rekap semua run

Bagian ini menampilkan hasil validasi semua run yang ada di folder hasil, termasuk waktu training 3 epoch, memori puncak, dan latensi. Latensi adalah waktu yang dibutuhkan model untuk menebak, diukur dengan batch 1 (satu data sekaligus) dan batch 32 (dihitung per data).

In [3]:
runs = pd.read_csv(FULL_DIR / "runs.csv")
cols = ["run_id", "best_epoch", "val_accuracy", "val_precision", "val_recall", "val_f1", "val_macro_f1",
        "train_min_total", "peak_mem_gb", "latency_bs1_ms", "latency_bs32_ms"]
display(runs[cols].round(4))


,run_id,best_epoch,val_accuracy,val_precision,val_recall,val_f1,val_macro_f1,train_min_total,peak_mem_gb,latency_bs1_ms,latency_bs32_ms
0,P2_early_roberta_L0_lr1e-04_s42,3,0.6543,0.6613,0.6149,0.6372,0.6536,14.2867,0.7153,7.9119,0.6221
1,P2_early_roberta_L0_lr1e-03_s42,3,0.6562,0.6642,0.6146,0.6384,0.6554,14.2577,0.7156,7.7954,0.6199
2,P2_early_roberta_L3_lr1e-05_s42,3,0.7794,0.7771,0.7756,0.7764,0.7793,22.1726,1.6700,8.0189,0.6185
3,P2_early_roberta_L6_lr1e-05_s42,3,0.7901,0.7855,0.7910,0.7882,0.7901,31.3015,2.6992,8.1977,0.6212
4,P2_early_distilbert_L0_lr1e-04_s42,3,0.6510,0.6546,0.6206,0.6372,0.6505,8.0438,0.4667,3.9054,0.3216
5,P2_early_distilbert_L0_lr1e-03_s42,3,0.6500,0.6556,0.6133,0.6338,0.6493,8.0247,0.4667,3.9356,0.3210
6,P2_early_distilbert_L3_lr2e-05_s42,3,0.7675,0.7693,0.7558,0.7625,0.7674,16.2420,1.4166,4.0742,0.3215
7,S43_comment_roberta_lr2e-05_s43,3,0.7836,0.7793,0.7838,0.7815,0.7836,49.1000,5.0915,8.2954,0.2818
8,S43_comment_distilbert_lr2e-05_s43,2,0.7660,0.7578,0.7733,0.7655,0.7660,25.5186,2.6342,4.2064,0.1487
9,S43_early_roberta_lr1e-05_s43,3,0.7959,0.7874,0.8035,0.7954,0.7959,56.3571,5.1119,8.3501,0.6089


#### Insight
Folder hasil berisi 60 run, semuanya dari RTX 5070 Ti.

Macro-F1 validasi S1 (seed 42) untuk setiap learning rate:

| Input | Model | 1e-5 | 2e-5 | 3e-5 | 5e-5 |
|---|---|---:|---:|---:|---:|
| Comment saja | RoBERTa | 0,7835 | 0,7830 | 0,7799 | 0,7703 |
| Comment saja | DistilBERT | 0,7667 | 0,7670 | 0,7672 | 0,7637 |
| Early fusion | RoBERTa | 0,7955 | 0,7974 | 0,7936 | 0,7839 |
| Early fusion | DistilBERT | 0,7756 | 0,7752 | 0,7743 | 0,7687 |
| Late fusion | RoBERTa | 0,7917 | 0,7914 | 0,7892 | 0,7806 |
| Late fusion | DistilBERT | 0,7762 | 0,7758 | 0,7748 | 0,7695 |

RoBERTa cocok dengan learning rate kecil. Hasil terbaiknya selalu di 1e-5 atau 2e-5, sedangkan 5e-5 selalu paling rendah, sekitar 1,1 sampai 1,4 poin di bawah yang terbaik.

DistilBERT hampir tidak terpengaruh learning rate antara 1e-5 dan 3e-5, selisihnya paling besar 0,13 poin. Baru di 5e-5 hasilnya turun 0,35 sampai 0,7 poin.

Kolom `best_epoch` juga menunjukkan pola yang sama dengan notebook 05. Di S1, RoBERTa selalu terbaik di epoch 3, sedangkan DistilBERT dengan learning rate 2e-5 ke atas umumnya sudah terbaik di epoch 2. Learning rate yang besar membuat model lebih cepat overfitting.

Learning rate terbaik di GPU ini berbeda dari hasil di RTX 5060 Ti untuk 4 konfigurasi, tetapi selisihnya sangat kecil. Contohnya early fusion RoBERTa, 2e-5 mendapat 0,7974 dan 1e-5 mendapat 0,7955, selisih 0,19 poin. Untuk comment saja DistilBERT, selisih 3e-5 dan 2e-5 hanya 0,02 poin. Selisih sekecil ini masih dalam batas kebetulan, karena dengan 100.276 data validasi galat bakunya sekitar 0,13 poin. Learning rate terbaik tetap diambil dari nilai validasi tertinggi supaya aturannya konsisten.

## 8. Uji jumlah layer yang dibekukan (S2)

Bagian ini menampilkan hasil S2, yaitu percobaan membekukan sebagian layer. Layer yang dibekukan tidak ikut diubah saat training, jadi pengetahuan bawaan model dipakai apa adanya. Yang dibekukan selalu layer bagian bawah, sedangkan layer bagian atas tetap dilatih. Lapisan klasifikasi, yaitu lapisan kecil di paling atas yang mengubah hasil encoder menjadi tebakan sarkas atau bukan, selalu ikut dilatih.

RoBERTa punya 12 layer dan DistilBERT punya 6 layer. Variasi yang diuji:

| Model | Layer dibekukan | Layer dilatih | Kode di nama run |
|---|---|---|---|
| RoBERTa | 12 (semua) | 0 (hanya lapisan klasifikasi) | L0 |
| RoBERTa | 9 | 3 | L3 |
| RoBERTa | 6 | 6 | L6 |
| RoBERTa | 0 | 12 (training penuh, dari S1 dan S3) | |
| DistilBERT | 6 (semua) | 0 (hanya lapisan klasifikasi) | L0 |
| DistilBERT | 3 | 3 | L3 |
| DistilBERT | 0 | 6 (training penuh, dari S1 dan S3) | |

Cara membaca tabel di bawah. Satu baris adalah satu run, dengan nomor urut 1 sampai 16. Kolom `seed` menunjukkan seed yang dipakai. Kolom `keterangan` berisi salah satu dari tiga hal berikut:
- `dipakai`: run dengan learning rate terbaik early fusion (RoBERTa 2e-5, DistilBERT 1e-5), diulang dengan seed 42, 43, dan 44,
- `LR lama, tidak dipakai`: run awal yang memakai learning rate terbaik versi lama, disimpan sebagai catatan saja,
- `uji LR besar (head saja)`: run dengan semua layer dibekukan, dicoba dengan learning rate 1e-4 dan 1e-3 karena yang dilatih hanya lapisan klasifikasi yang kecil.

In [4]:
TOTAL_LAYER = {"roberta": 12, "distilbert": 6}

p2 = runs[runs["round"] == "P2"].copy()
p2["layer_dilatih"] = p2["n_train_layers"].astype(int)
p2["layer_dibekukan"] = p2["model"].map(TOTAL_LAYER) - p2["layer_dilatih"]
lr_terbaik = p2["model"].map(lambda m: BEST_LR[("early", m)])
p2["keterangan"] = "dipakai"
p2.loc[p2["layer_dilatih"] == 0, "keterangan"] = "uji LR besar (head saja)"
p2.loc[(p2["layer_dilatih"] > 0) & (p2["lr"] != lr_terbaik), "keterangan"] = "LR lama, tidak dipakai"
p2["lr"] = p2["lr"].map(lambda x: f"{x:.0e}")          # tampil 1e-05, bukan 0.0000

cols = ["run_id", "model", "layer_dibekukan", "layer_dilatih", "lr", "seed", "keterangan", "best_epoch",
        "val_accuracy", "val_precision", "val_recall", "val_f1", "val_macro_f1",
        "params_trainable_m", "train_min_per_epoch", "peak_mem_gb", "latency_bs1_ms"]
p2_tabel = (p2.sort_values(["model", "layer_dibekukan", "keterangan", "lr", "seed"],
                           ascending=[False, False, True, True, True])[cols].reset_index(drop=True))
p2_tabel.index += 1                                     # nomor urut mulai dari 1
display(p2_tabel.round(4))

,run_id,model,layer_dibekukan,layer_dilatih,lr,seed,keterangan,best_epoch,val_accuracy,val_precision,val_recall,val_f1,val_macro_f1,params_trainable_m,train_min_per_epoch,peak_mem_gb,latency_bs1_ms
1,P2_early_roberta_L0_lr1e-03_s42,roberta,12,0,1e-03,42,uji LR besar (head saja),3,0.6562,0.6642,0.6146,0.6384,0.6554,0.5921,4.7526,0.7156,7.7954
2,P2_early_roberta_L0_lr1e-04_s42,roberta,12,0,1e-04,42,uji LR besar (head saja),3,0.6543,0.6613,0.6149,0.6372,0.6536,0.5921,4.7622,0.7153,7.9119
3,P2_early_roberta_L3_lr1e-05_s42,roberta,9,3,1e-05,42,"LR lama, tidak dipakai",3,0.7794,0.7771,0.7756,0.7764,0.7793,21.8557,7.3909,1.6700,8.0189
4,P2_early_roberta_L3_lr2e-05_s42,roberta,9,3,2e-05,42,dipakai,3,0.7863,0.7844,0.7823,0.7833,0.7862,21.8557,7.5341,1.6697,8.0147
5,P2_early_roberta_L3_lr2e-05_s43,roberta,9,3,2e-05,43,dipakai,3,0.7863,0.7861,0.7792,0.7827,0.7862,21.8557,7.4848,1.6705,8.1484
6,P2_early_roberta_L3_lr2e-05_s44,roberta,9,3,2e-05,44,dipakai,3,0.7860,0.7850,0.7804,0.7827,0.7860,21.8557,7.4603,1.6706,8.0475
7,P2_early_roberta_L6_lr1e-05_s42,roberta,6,6,1e-05,42,"LR lama, tidak dipakai",3,0.7901,0.7855,0.7910,0.7882,0.7901,43.1194,10.4338,2.6992,8.1977
8,P2_early_roberta_L6_lr2e-05_s42,roberta,6,6,2e-05,42,dipakai,3,0.7943,0.7890,0.7964,0.7927,0.7943,43.1194,10.4939,2.7003,8.1253
9,P2_early_roberta_L6_lr2e-05_s43,roberta,6,6,2e-05,43,dipakai,3,0.7943,0.7930,0.7897,0.7913,0.7943,43.1194,10.5263,2.7014,8.2339
10,P2_early_roberta_L6_lr2e-05_s44,roberta,6,6,2e-05,44,dipakai,3,0.7948,0.7913,0.7937,0.7925,0.7947,43.1194,10.5041,2.7005,8.2026


#### Insight
Hasil S2 dibandingkan dengan early fusion yang dilatih penuh (seed 42, dari bagian 7):

| Model | Layer dibekukan | Layer dilatih | Parameter dilatih | Macro-F1 | Menit per epoch | Memori (GB) | Latensi batch 1 (ms) |
|---|---:|---|---:|---:|---:|---:|---:|
| RoBERTa | 12 | 0 | 0,59 juta | 0,6554 | 4,8 | 0,72 | 7,8 |
| RoBERTa | 9 | 3 | 21,86 juta | 0,7862 | 7,5 | 1,67 | 8,0 |
| RoBERTa | 6 | 6 | 43,12 juta | 0,7943 | 10,5 | 2,70 | 8,1 |
| RoBERTa | 0 | 12 (penuh) | 124,6 juta | 0,7974 | 18,9 | 5,11 | 8,4 |
| DistilBERT | 6 | 0 | 0,59 juta | 0,6505 | 2,7 | 0,47 | 3,9 |
| DistilBERT | 3 | 3 | 21,86 juta | 0,7611 | 5,5 | 1,41 | 4,1 |
| DistilBERT | 0 | 6 (penuh) | 65,8 juta | 0,7756 | 9,9 | 2,64 | 4,2 |

Kalau semua layer dibekukan dan hanya lapisan klasifikasi yang dilatih, macro-F1 hanya sekitar 0,65. Encoder bawaan belum cukup untuk mengenali sarkasme tanpa ikut disesuaikan.

Membekukan setengah layer RoBERTa (6 dibekukan, 6 dilatih) hanya 0,3 poin di bawah RoBERTa penuh, padahal waktu training-nya sekitar 56% dan memorinya sekitar 53%. RoBERTa dengan 9 layer dibekukan dan hanya 3 layer dilatih (0,7862) bahkan lebih tinggi dari DistilBERT yang dilatih penuh (0,7756), dengan waktu training dan memori yang lebih kecil.

Membekukan layer tidak mempercepat waktu menebak. Latensi RoBERTa tetap sekitar 8 ms dan DistilBERT sekitar 4 ms, karena saat menebak data tetap harus melewati semua layer. Penghematannya hanya terjadi saat training.

Tiga seed S2 menghasilkan nilai yang hampir sama, misalnya RoBERTa 3 layer 0,7862, 0,7862, dan 0,7860. Seed-nya memang berbeda, terlihat dari log bagian 6 yang nilai epoch 1-nya tidak sama. Model yang sebagian besar layernya dibekukan hanya punya sedikit bagian yang bisa berubah, jadi hasil akhirnya lebih stabil.

Run S2 lama yang memakai learning rate lain memberi petunjuk tambahan. RoBERTa 3 layer mendapat 0,7793 dengan 1e-5 dan 0,7862 dengan 2e-5, RoBERTa 6 layer 0,7901 dengan 1e-5 dan 0,7943 dengan 2e-5, sedangkan DistilBERT 3 layer mendapat 0,7674 dengan 2e-5 dan 0,7611 dengan 1e-5. Jadi model dengan layer yang dibekukan cenderung lebih baik dengan learning rate yang lebih besar. Karena S2 memakai learning rate terbaik dari training penuh, hasil DistilBERT 3 layer kemungkinan sedikit di bawah potensinya. Ini perlu ditulis sebagai keterbatasan.

## 9. Tabel gabungan seed versi awal

Bagian ini adalah tabel gabungan seed yang dibuat sebelum S1 diulang di RTX 5070 Ti. Di tabel ini seed 42 masih diambil dari RTX 5060 Ti, dan karena learning rate terbaik kemudian berubah, beberapa konfigurasi muncul dengan dua learning rate. Sel ini dibiarkan sebagai catatan saja. Angka final ada di bagian 10.

In [19]:
runs_old = pd.read_csv(ROOT / "data/training_rtx5060ti/full/runs.csv")
seeds = pd.concat([runs_old.merge(runs[runs["round"] == "S43"][["input", "model", "lr"]]),
                   runs[runs["round"].isin(["S43", "S44"])]])
metrics = ["val_accuracy", "val_precision", "val_recall", "val_f1", "val_macro_f1"]
display(seeds.pivot_table(index=["input", "model", "lr"], columns="seed", values="val_macro_f1").round(4))
g = seeds.groupby(["input", "model"])[metrics]
display((g.mean() * 100).round(2).astype(str) + " ± " + (g.std() * 100).round(2).astype(str))


seed                            42      43      44
input   model      lr                             
comment distilbert 0.00002  0.7677  0.7660  0.7666
                   0.00003  0.7667  0.7655  0.7654
        roberta    0.00001  0.7827  0.7829  0.7836
                   0.00002  0.7834  0.7836  0.7844
early   distilbert 0.00001  0.7731  0.7749  0.7743
                   0.00002  0.7744  0.7755  0.7755
        roberta    0.00001  0.7962  0.7959  0.7961
                   0.00002  0.7950  0.7968  0.7965
late    distilbert 0.00001  0.7767  0.7770  0.7765
        roberta    0.00001  0.7918  0.7939  0.7914

val_accuracy val_precision    val_recall        val_f1  \
input   model                                                                
comment distilbert   76.65 ± 0.1  76.85 ± 0.93  75.47 ± 1.44  76.14 ± 0.29   
        roberta     78.35 ± 0.06   78.13 ± 0.3  77.99 ± 0.43  78.06 ± 0.08   
early   distilbert   77.48 ± 0.1  77.87 ± 0.97  76.04 ± 1.55  76.93 ± 0.35   
        roberta     79.61 ± 0.06  79.17 ± 0.95  79.73 ± 1.59  79.43 ± 0.33   
late    distilbert  77.68 ± 0.02  77.75 ± 0.07  76.77 ± 0.16  77.26 ± 0.05   
        roberta     79.24 ± 0.13    78.7 ± 0.4  79.47 ± 0.37  79.08 ± 0.06   

                    val_macro_f1  
input   model                     
comment distilbert  76.63 ± 0.09  
        roberta     78.34 ± 0.06  
early   distilbert  77.46 ± 0.09  
        roberta     79.61 ± 0.06  
late    distilbert  77.67 ± 0.02  
        roberta     79.24 ± 0.13

#### Insight
Untuk learning rate yang sama, seed 42 dari RTX 5060 Ti dan seed 43 serta 44 dari RTX 5070 Ti hasilnya sangat dekat. Contohnya early fusion RoBERTa dengan 1e-5 mendapat 0,7962, 0,7959, dan 0,7961, sedangkan late fusion RoBERTa mendapat 0,7918, 0,7939, dan 0,7914. Jadi pindah GPU hampir tidak mengubah akurasi.

Tabel mean ± std di bawahnya tidak dipakai, karena untuk comment saja dan early fusion isinya mencampur dua learning rate dan dua GPU.

## 10. Rekap final (semua dari RTX 5070 Ti)

Bagian ini menghitung ulang hasil final hanya dari run di GPU ini. Langkahnya: learning rate terbaik diambil dari S1 seed 42, konfigurasi itu digabung dengan seed 43 dan 44, lalu dihitung rata-rata dan simpangan baku (std). S2 juga dihitung dengan 3 seed. Terakhir, efisiensi dirata-rata dari semua run S1 dan S3, karena learning rate dan seed tidak memengaruhi kecepatan.

In [6]:
runs = pd.read_csv(FULL_DIR / "runs.csv")
runs["putaran"] = runs["round"].astype(str)
M = ["val_accuracy", "val_precision", "val_recall", "val_f1", "val_macro_f1"]
print("Jumlah run:", len(runs), "| GPU:", runs["gpu"].unique(), "| run_id kembar:", runs["run_id"].duplicated().sum())

# 1) Pencarian LR (seed 42): macro-F1 per LR dan LR terbaik
s1 = runs[runs["putaran"].isin(["1", "2", "3"])]
display(s1.pivot_table(index=["input", "model"], columns="lr", values="val_macro_f1").mul(100).round(2))
best = s1.loc[s1.groupby(["input", "model"])["val_macro_f1"].idxmax(), ["input", "model", "lr"]]
display(best)

# 2) Konfigurasi terbaik x 3 seed (42, 43, 44): mean ± std
main = pd.concat([s1, runs[runs["putaran"].isin(["S43", "S44"])]]).merge(best, on=["input", "model", "lr"])
display(main.pivot_table(index=["input", "model"], columns="seed", values="val_macro_f1").mul(100).round(2))
g = main.groupby(["input", "model"])
display((g[M].mean() * 100).round(2).astype(str) + " ± " + (g[M].std() * 100).round(2).astype(str))

# 3) Jumlah layer (S2) x 3 seed, LR = LR terbaik early fusion
lr_early = best[best["input"] == "early"].set_index("model")["lr"]
p2 = runs[(runs["putaran"] == "P2") & (runs["n_train_layers"].astype(str) != "0")]
p2 = p2[p2["lr"] == p2["model"].map(lr_early)]
display(p2.pivot_table(index=["model", "n_train_layers"], columns="seed", values="val_macro_f1").mul(100).round(2))
display(p2.groupby(["model", "n_train_layers"])[["train_min_per_epoch", "peak_mem_gb", "latency_bs1_ms"]].mean().round(2))

# 4) Efisiensi (semua run S1 + seed, LR tidak memengaruhi kecepatan)
E = ["train_min_per_epoch", "latency_bs1_ms", "latency_bs32_ms", "peak_mem_gb", "params_total_m"]
eff = runs[runs["putaran"].isin(["1", "2", "3", "S43", "S44"])].groupby(["input", "model"])[E].mean()
display(eff.round(3))
display(pd.DataFrame({i: eff.loc[(i, "roberta")] / eff.loc[(i, "distilbert")] for i in ["comment", "early", "late"]}).T.round(2))

Jumlah run: 60 | GPU: ['NVIDIA GeForce RTX 5070 Ti'] | run_id kembar: 0


lr                  0.00001  0.00002  0.00003  0.00005
input   model                                         
comment distilbert    76.67    76.70    76.72    76.37
        roberta       78.35    78.30    77.99    77.03
early   distilbert    77.56    77.52    77.43    76.87
        roberta       79.55    79.74    79.36    78.39
late    distilbert    77.62    77.58    77.48    76.95
        roberta       79.17    79.14    78.92    78.06

,input,model,lr
20,comment,distilbert,0.00003
31,comment,roberta,0.00001
26,early,distilbert,0.00001
27,early,roberta,0.00002
38,late,distilbert,0.00001
37,late,roberta,0.00001


seed                   42     43     44
input   model                          
comment distilbert  76.72  76.55  76.54
        roberta     78.35  78.29  78.36
early   distilbert  77.56  77.49  77.43
        roberta     79.74  79.68  79.65
late    distilbert  77.62  77.70  77.65
        roberta     79.17  79.39  79.14

val_accuracy val_precision    val_recall        val_f1  \
input   model                                                                
comment distilbert  76.61 ± 0.11  76.71 ± 0.75   75.61 ± 1.1   76.15 ± 0.2   
        roberta     78.34 ± 0.04  78.15 ± 0.33  77.92 ± 0.52   78.04 ± 0.1   
early   distilbert  77.52 ± 0.05  78.05 ± 1.12  75.83 ± 1.97  76.91 ± 0.49   
        roberta      79.7 ± 0.03   79.37 ± 1.3  79.64 ± 2.22  79.48 ± 0.48   
late    distilbert  77.66 ± 0.04  77.65 ± 0.11  76.89 ± 0.15  77.27 ± 0.05   
        roberta     79.24 ± 0.13  78.74 ± 0.36  79.38 ± 0.25  79.06 ± 0.06   

                    val_macro_f1  
input   model                     
comment distilbert    76.6 ± 0.1  
        roberta     78.34 ± 0.04  
early   distilbert   77.5 ± 0.07  
        roberta     79.69 ± 0.04  
late    distilbert  77.65 ± 0.04  
        roberta     79.23 ± 0.13

seed                          42     43     44
model      n_train_layers                     
distilbert 3               76.11  76.11  76.15
roberta    3               78.62  78.62  78.60
           6               79.43  79.43  79.47

train_min_per_epoch  peak_mem_gb  latency_bs1_ms
model      n_train_layers                                                  
distilbert 3                              5.47         1.41            4.10
roberta    3                              7.49         1.67            8.07
           6                             10.51         2.70            8.19

train_min_per_epoch  latency_bs1_ms  latency_bs32_ms  \
input   model                                                              
comment distilbert                8.553           4.222            0.150   
        roberta                  16.506           8.396            0.298   
early   distilbert                9.883           4.206            0.317   
        roberta                  18.883           8.389            0.609   
late    distilbert               14.955           7.284            0.428   
        roberta                  28.872          15.136            0.839   

                    peak_mem_gb  params_total_m  
input   model                                    
comment distilbert        2.632          65.783  
        roberta           5.087         124.647  
early   distilbert        2.640          65.783  
        roberta           5.107         124.647  
late    distilbert        4.286          67.553  
        roberta           8.383         127.007

,train_min_per_epoch,latency_bs1_ms,latency_bs32_ms,peak_mem_gb,params_total_m
comment,1.93,1.99,1.98,1.93,1.89
early,1.91,1.99,1.92,1.93,1.89
late,1.93,2.08,1.96,1.96,1.88


#### Insight
Semua 60 run berasal dari RTX 5070 Ti dan tidak ada nama run yang kembar.

Learning rate terbaik dari S1: comment saja RoBERTa 1e-5, comment saja DistilBERT 3e-5, early fusion RoBERTa 2e-5, early fusion DistilBERT 1e-5, late fusion RoBERTa dan DistilBERT 1e-5.

Hasil validasi konfigurasi terbaik, rata-rata ± std dari seed 42, 43, dan 44 (dalam persen):

| Input | Model | Accuracy | Precision | Recall | F1 | Macro-F1 |
|---|---|---:|---:|---:|---:|---:|
| Comment saja | RoBERTa | 78,34 ± 0,04 | 78,15 ± 0,33 | 77,92 ± 0,52 | 78,04 ± 0,10 | 78,34 ± 0,04 |
| Comment saja | DistilBERT | 76,61 ± 0,11 | 76,71 ± 0,75 | 75,61 ± 1,10 | 76,15 ± 0,20 | 76,60 ± 0,10 |
| Early fusion | RoBERTa | 79,70 ± 0,03 | 79,37 ± 1,30 | 79,64 ± 2,22 | 79,48 ± 0,48 | 79,69 ± 0,04 |
| Early fusion | DistilBERT | 77,52 ± 0,05 | 78,05 ± 1,12 | 75,83 ± 1,97 | 76,91 ± 0,49 | 77,50 ± 0,07 |
| Late fusion | RoBERTa | 79,24 ± 0,13 | 78,74 ± 0,36 | 79,38 ± 0,25 | 79,06 ± 0,06 | 79,23 ± 0,13 |
| Late fusion | DistilBERT | 77,66 ± 0,04 | 77,65 ± 0,11 | 76,89 ± 0,15 | 77,27 ± 0,05 | 77,65 ± 0,04 |

Hasilnya sangat stabil. Macro-F1 antar-seed hanya bergeser 0,04 sampai 0,13 poin. Precision dan recall early fusion bergeser lebih besar (sampai 2,22 poin), artinya di satu seed model lebih sering menebak sarkas dan di seed lain lebih jarang, tetapi kemampuan totalnya tetap sama. Ini juga alasan epoch terbaik dipilih dengan macro-F1.

RoBERTa selalu lebih tinggi dari DistilBERT di setiap seed: 1,73 poin untuk comment saja, 2,19 poin untuk early fusion, dan 1,58 poin untuk late fusion. Konteks parent juga selalu membantu. Early fusion menaikkan macro-F1 RoBERTa 1,35 poin dan DistilBERT 0,89 poin dibanding comment saja. Late fusion RoBERTa justru 0,46 poin di bawah early fusion di ketiga seed, sedangkan pada DistilBERT keduanya hampir sama (late fusion 0,16 poin di atas).

S2 dengan 3 seed menghasilkan RoBERTa 3 layer 78,61, RoBERTa 6 layer 79,44, dan DistilBERT 3 layer 76,12. RoBERTa 3 layer tetap 1,11 poin di atas DistilBERT penuh (77,50).

Efisiensi (rata-rata semua run S1 dan S3):

| Input | Model | Menit per epoch | Latensi batch 1 (ms) | Latensi batch 32 (ms per data) | Memori (GB) | Parameter (juta) |
|---|---|---:|---:|---:|---:|---:|
| Comment saja | RoBERTa | 16,51 | 8,40 | 0,30 | 5,09 | 124,6 |
| Comment saja | DistilBERT | 8,55 | 4,22 | 0,15 | 2,63 | 65,8 |
| Early fusion | RoBERTa | 18,88 | 8,39 | 0,61 | 5,11 | 124,6 |
| Early fusion | DistilBERT | 9,88 | 4,21 | 0,32 | 2,64 | 65,8 |
| Late fusion | RoBERTa | 28,87 | 15,14 | 0,84 | 8,38 | 127,0 |
| Late fusion | DistilBERT | 14,96 | 7,28 | 0,43 | 4,29 | 67,6 |

Rasio RoBERTa dibanding DistilBERT hampir sama di ketiga input: waktu training 1,91 sampai 1,93 kali, latensi batch 1 1,99 sampai 2,08 kali, memori 1,93 sampai 1,96 kali, dan jumlah parameter 1,88 sampai 1,89 kali. Rasio ini juga sama dengan yang didapat di RTX 5060 Ti, jadi kesimpulan efisiensinya tidak bergantung pada GPU.

Early fusion hanya 1,14 sampai 1,16 kali lebih lama dari comment saja, dan latensi batch 1 serta memorinya sama. Yang naik sekitar 2 kali hanya latensi batch 32, karena inputnya memang lebih panjang. Late fusion sekitar 1,5 kali lebih lama dari early fusion, latensi batch 1-nya sekitar 1,75 kali, dan memorinya sekitar 1,6 kali, karena encoder dijalankan dua kali (sekali untuk parent, sekali untuk comment).

## 11. Kesimpulan

1. Semua training final (60 run) selesai di satu GPU, RTX 5070 Ti. Hasil akhir memakai 24 run S1, 13 run S2, dan 12 run seed tambahan. Run dengan learning rate lama tetap disimpan tetapi tidak dipakai.
2. RoBERTa paling baik dengan learning rate 1e-5 atau 2e-5, dan 5e-5 terlalu besar. DistilBERT hampir tidak terpengaruh learning rate kecuali di 5e-5.
3. Konfigurasi terbaik di validasi adalah early fusion RoBERTa dengan macro-F1 79,69 ± 0,04.
4. RoBERTa 1,6 sampai 2,2 poin lebih tinggi dari DistilBERT, tetapi butuh sekitar 1,9 kali waktu training, sekitar 2 kali waktu menebak, dan sekitar 1,9 kali memori.
5. Konteks parent membantu, dan early fusion adalah cara termurah untuk memasukkannya. Late fusion tidak lebih baik dari early fusion tetapi jauh lebih mahal.
6. Melatih sebagian layer RoBERTa menghemat biaya training. Dengan 3 layer, RoBERTa sudah lebih tinggi dari DistilBERT penuh, tetapi waktu menebaknya tetap sekitar 2 kali DistilBERT.

Hasil di notebook ini masih hasil validasi. Penilaian akhir dengan data test dan uji signifikansi ada di notebook 07.